# Board Reconstruction - Métrica NeurIPS 2024

Implementación de la métrica **Reconstruction** del paper:
"Measuring Progress in Dictionary Learning for Language Model Interpretability with Board Game Models"
(Karvonen et al., NeurIPS 2024)

## Algoritmo:

**Fase 1 - Training Set:**
1. Para cada feature del SAE y cada BSP:
   - Calcular precisión de la feature para detectar esa BSP
   - Si precisión ≥ 0.95 → guardar como "feature de alta confianza"

**Fase 2 - Test Set:**
1. Para cada posición:
   - Si alguna feature de alta confianza para BSP_X se activa → predecir BSP_X = True
   - Si ninguna se activa → predecir BSP_X = False (vacía)
2. Calcular F1 **solo sobre casillas con piezas** (no puntúa vacías)

**Objetivo del paper:** Reconstruction ~0.95 para Othello

In [21]:
import numpy as np
import torch
import sys
from pathlib import Path
from tqdm import tqdm

# Proyecto root
project_root = Path('../..').resolve()
sys.path.insert(0, str(project_root))

print(f"Proyecto: {project_root}")

Proyecto: C:\Users\Esposa\Documents\Repos\othello_world


## 1. Cargar Datos y Modelo

In [2]:
# Cargar activaciones
activations_path = project_root / "sae" / "activations" / "data" / "layer5_200games.npy"
activations = np.load(activations_path)

print(f"Activaciones cargadas:")
print(f"  Shape: {activations.shape}")
print(f"  Memoria: {activations.nbytes / (1024**2):.2f} MB")

Activaciones cargadas:
  Shape: (11800, 512)
  Memoria: 23.05 MB


In [3]:
# Cargar ground truth de BSPs
bsp_gt_path = project_root / "sae" / "metrics" / "data" / "bsp_ground_truth_200games.npy"
bsp_names_path = project_root / "sae" / "metrics" / "data" / "bsp_ground_truth_200games.names.npy"

bsp_ground_truth = np.load(bsp_gt_path)
bsp_names = np.load(bsp_names_path, allow_pickle=True)

print(f"Ground truth BSPs:")
print(f"  Shape: {bsp_ground_truth.shape}")
print(f"  Total BSPs: {len(bsp_names)}")

Ground truth BSPs:
  Shape: (11800, 198)
  Total BSPs: 198


In [4]:
# Cargar modelo SAE
from sae.model.sae import SparseAutoencoder

input_dim = 512
hidden_dim = 16384

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sae = SparseAutoencoder(input_dim, hidden_dim).to(device)

model_path = project_root / "sae" / "model" / "saved_model" / "sae_othello_best.pt"
checkpoint = torch.load(model_path, map_location=device)
sae.load_state_dict(checkpoint['model_state_dict'])
sae.eval()

print(f"SAE cargado:")
print(f"  Device: {device}")
print(f"  Expansion: {hidden_dim/input_dim}x")

SAE cargado:
  Device: cuda
  Expansion: 32.0x


In [5]:
# Extraer features del SAE
activations_tensor = torch.from_numpy(activations).float().to(device)

with torch.no_grad():
    encoded = torch.relu(sae.encoder(activations_tensor))
    
sae_features = encoded.cpu().numpy()

print(f"Features SAE extraídas:")
print(f"  Shape: {sae_features.shape}")
print(f"  Sparsity: {np.mean(sae_features == 0):.2%}")

Features SAE extraídas:
  Shape: (11800, 16384)
  Sparsity: 96.54%


## 2. Split Train/Test

Usaremos:
- **Train:** Primeras 100 partidas (5,900 posiciones)
- **Test:** Últimas 100 partidas (5,900 posiciones)

In [6]:
# Split datos
n_games = 200
n_moves = 59
split_game = 100

split_idx = split_game * n_moves

# Train set
sae_features_train = sae_features[:split_idx]
bsp_gt_train = bsp_ground_truth[:split_idx]

# Test set
sae_features_test = sae_features[split_idx:]
bsp_gt_test = bsp_ground_truth[split_idx:]

print(f"Train set:")
print(f"  Features: {sae_features_train.shape}")
print(f"  BSPs: {bsp_gt_train.shape}")
print()
print(f"Test set:")
print(f"  Features: {sae_features_test.shape}")
print(f"  BSPs: {bsp_gt_test.shape}")

Train set:
  Features: (5900, 16384)
  BSPs: (5900, 198)

Test set:
  Features: (5900, 16384)
  BSPs: (5900, 198)


## 3. Funciones Optimizadas

Versiones vectorizadas de precision y F1-score.

In [7]:
def fast_precision_score(y_true, y_pred):
    """
    Precisión vectorizada sin sklearn.
    ~10x más rápido para arrays binarios.
    """
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)
    
    tp = np.sum(y_true & y_pred)
    fp = np.sum(~y_true & y_pred)
    
    if tp == 0:
        return 0.0
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    return precision


def fast_f1_score(y_true, y_pred):
    """
    F1-score vectorizado sin sklearn.
    ~10x más rápido para arrays binarios.
    """
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)
    
    tp = np.sum(y_true & y_pred)
    fp = np.sum(~y_true & y_pred)
    fn = np.sum(y_true & ~y_pred)
    
    if tp == 0:
        return 0.0
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    if precision + recall == 0:
        return 0.0
    
    return 2 * (precision * recall) / (precision + recall)

## 4. Filtrar BSPs de Piezas

Reconstruction solo evalúa las 128 BSPs de piezas (igual que Coverage).

In [8]:
# Filtrar solo BSPs de piezas (sin vacías, sin especiales)
bsp_pieces_indices = []
bsp_pieces_names = []

for i, name in enumerate(bsp_names):
    if len(name) == 6 and name.startswith('BSP') and not name.endswith('0'):
        bsp_pieces_indices.append(i)
        bsp_pieces_names.append(name)

bsp_pieces_indices = np.array(bsp_pieces_indices)

# Aplicar filtro a train y test
bsp_gt_train_pieces = bsp_gt_train[:, bsp_pieces_indices]
bsp_gt_test_pieces = bsp_gt_test[:, bsp_pieces_indices]

print(f"BSPs para Reconstruction:")
print(f"  Total BSPs de piezas: {len(bsp_pieces_indices)}")
print(f"  Train shape: {bsp_gt_train_pieces.shape}")
print(f"  Test shape: {bsp_gt_test_pieces.shape}")

BSPs para Reconstruction:
  Total BSPs de piezas: 128
  Train shape: (5900, 128)
  Test shape: (5900, 128)


## 4. Fase 1: Identificar Features de Alta Precisión (Train Set)

Para cada BSP, encontrar features del SAE con precisión ≥ 0.95.

In [9]:
def identify_high_precision_features(sae_features_train, bsp_gt_train, 
                                      precision_threshold=0.95,
                                      thresholds=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]):
    """
    Identifica features con precisión ≥ threshold para cada BSP (OPTIMIZADO).
    
    Args:
        sae_features_train: (n_train, n_features)
        bsp_gt_train: (n_train, n_bsps)
        precision_threshold: Mínima precisión requerida (default 0.95)
        thresholds: Umbrales de activación a probar
    
    Returns:
        high_precision_features: dict {bsp_idx: [(feature_idx, threshold), ...]}
    """
    n_positions, n_features = sae_features_train.shape
    n_bsps = bsp_gt_train.shape[1]
    
    print(f"Identificando features de alta precisión (OPTIMIZADO)...")
    print(f"  Threshold de precisión: {precision_threshold}")
    print(f"  BSPs a procesar: {n_bsps}")
    print(f"  Features del SAE: {n_features}")
    print()
    
    # Pre-calcular máximos
    print("Pre-calculando máximos de features...")
    f_max_per_feature = np.max(sae_features_train, axis=0)
    active_features = np.where(f_max_per_feature > 0)[0]
    print(f"✓ Máximos calculados. Features activas: {len(active_features)}/{n_features}")
    print()
    
    high_precision_features = {}
    
    for bsp_idx in tqdm(range(n_bsps), desc="Procesando BSPs"):
        bsp_labels = bsp_gt_train[:, bsp_idx].astype(bool)
        
        # Skip si nunca está activa
        if not bsp_labels.any():
            high_precision_features[bsp_idx] = []
            continue
        
        hp_features_for_bsp = []
        
        # Probar cada feature activa
        for feature_idx in active_features:
            feature_acts = sae_features_train[:, feature_idx]
            f_max = f_max_per_feature[feature_idx]
            
            # Probar cada threshold
            for t in thresholds:
                predictions = (feature_acts > t * f_max)
                
                # Calcular precisión (vectorizado)
                if predictions.any():
                    precision = fast_precision_score(bsp_labels, predictions)
                    
                    # Si cumple threshold, guardar
                    if precision >= precision_threshold:
                        hp_features_for_bsp.append((feature_idx, t))
                        break  # No probar más thresholds para esta feature
        
        high_precision_features[bsp_idx] = hp_features_for_bsp
    
    return high_precision_features

In [10]:
# Ejecutar fase 1
high_precision_features = identify_high_precision_features(
    sae_features_train,
    bsp_gt_train_pieces,
    precision_threshold=0.95
)

# Estadísticas
n_features_per_bsp = [len(features) for features in high_precision_features.values()]

print("\n" + "="*60)
print("FASE 1: Features de alta precisión identificadas")
print("="*60)
print(f"Total BSPs: {len(high_precision_features)}")
print(f"Features promedio por BSP: {np.mean(n_features_per_bsp):.1f}")
print(f"BSPs con 0 features: {np.sum(np.array(n_features_per_bsp) == 0)}")
print(f"BSPs con 1+ features: {np.sum(np.array(n_features_per_bsp) > 0)}")
print(f"BSPs con 5+ features: {np.sum(np.array(n_features_per_bsp) >= 5)}")
print("="*60)

Identificando features de alta precisión (OPTIMIZADO)...
  Threshold de precisión: 0.95
  BSPs a procesar: 128
  Features del SAE: 16384

Pre-calculando máximos de features...
✓ Máximos calculados. Features activas: 7775/16384



Procesando BSPs: 100%|██████████| 128/128 [11:25<00:00,  5.36s/it]


FASE 1: Features de alta precisión identificadas
Total BSPs: 128
Features promedio por BSP: 1889.9
BSPs con 0 features: 0
BSPs con 1+ features: 128
BSPs con 5+ features: 128


## 5. Fase 2: Reconstruir Tableros (Test Set)

Usar las features de alta precisión para predecir BSPs en test set.

In [16]:
def reconstruct_boards(sae_features_test, bsp_gt_test, high_precision_features, sae_features_train):
    """
    Reconstruye tableros usando features de alta precisión (OPTIMIZADO).
    
    Args:
        sae_features_test: (n_test, n_features)
        bsp_gt_test: (n_test, n_bsps) - ground truth para calcular F1
        high_precision_features: dict {bsp_idx: [(feature_idx, threshold), ...]}
        sae_features_train: Para calcular f_max
    
    Returns:
        reconstruction_score: F1 promedio sobre test set
        f1_per_position: Lista de F1 por posición
    """
    n_positions, n_features = sae_features_test.shape
    n_bsps = bsp_gt_test.shape[1]
    
    print(f"Reconstruyendo tableros en test set (OPTIMIZADO)...")
    print(f"  Posiciones: {n_positions}")
    print(f"  BSPs: {n_bsps}")
    print()
    
    # Pre-calcular máximos (del train set)
    print("Pre-calculando máximos de features...")
    f_max_per_feature = np.max(sae_features_train, axis=0)
    print("✓ Máximos calculados.")
    print()
    
    f1_per_position = []
    
    for pos_idx in tqdm(range(n_positions), desc="Reconstruyendo posiciones"):
        # Features activas en esta posición
        position_features = sae_features_test[pos_idx]
        
        # Predicciones para cada BSP
        predictions = np.zeros(n_bsps, dtype=bool)
        ground_truth = bsp_gt_test[pos_idx].astype(bool)
        
        # Para cada BSP, ver si alguna feature de alta confianza se activa
        for bsp_idx, hp_features in high_precision_features.items():
            for feature_idx, threshold in hp_features:
                f_max = f_max_per_feature[feature_idx]
                
                if position_features[feature_idx] > threshold * f_max:
                    predictions[bsp_idx] = True
                    break  # Ya encontramos una feature activa para esta BSP
        
        # Calcular F1 SOLO sobre casillas con piezas (vectorizado)
        # Si hay predicciones o ground truth activos
        if predictions.any() or ground_truth.any():
            f1 = fast_f1_score(ground_truth, predictions)
        else:
            f1 = 1.0  # Si ambos son todo False, es perfecto
        
        f1_per_position.append(f1)
    
    # Promedio de F1 sobre todas las posiciones
    reconstruction_score = np.mean(f1_per_position)
    
    return reconstruction_score, f1_per_position

In [ ]:
# Ejecutar fase 2
reconstruction_score, f1_per_position = reconstruct_boards(
    sae_features_test,
    bsp_gt_test_pieces,
    high_precision_features,
    sae_features_train
)

print("\n" + "="*60)
print("RESULTADO RECONSTRUCTION")
print("="*60)
print(f"Reconstruction Score: {reconstruction_score:.4f}")
print(f"Objetivo (paper): 0.95")
print(f"Diferencia: {reconstruction_score - 0.95:.4f}")
print("="*60)


Reconstruyendo tableros en test set (OPTIMIZADO)...
  Posiciones: 5900
  BSPs: 128

Pre-calculando máximos de features...
✓ Máximos calculados.



Reconstruyendo posiciones: 100%|██████████| 5900/5900 [05:39<00:00, 17.36it/s]



RESULTADO RECONSTRUCTION
Reconstruction Score: 0.2292
Objetivo (paper): 0.95
Diferencia: -0.7208


## 6. Análisis de Resultados

In [18]:
# Distribución de F1 scores
f1_array = np.array(f1_per_position)

print("Distribución de F1 por posición:")
print(f"  Mínimo: {np.min(f1_array):.4f}")
print(f"  Máximo: {np.max(f1_array):.4f}")
print(f"  Media: {np.mean(f1_array):.4f}")
print(f"  Mediana: {np.median(f1_array):.4f}")
print(f"  Std: {np.std(f1_array):.4f}")
print()
print(f"Posiciones con F1 > 0.95: {np.sum(f1_array > 0.95)} ({np.mean(f1_array > 0.95):.1%})")
print(f"Posiciones con F1 > 0.90: {np.sum(f1_array > 0.90)} ({np.mean(f1_array > 0.90):.1%})")
print(f"Posiciones con F1 < 0.80: {np.sum(f1_array < 0.80)} ({np.mean(f1_array < 0.80):.1%})")

Distribución de F1 por posición:
  Mínimo: 0.0000
  Máximo: 1.0000
  Media: 0.2292
  Mediana: 0.1176
  Std: 0.2549

Posiciones con F1 > 0.95: 285 (4.8%)
Posiciones con F1 > 0.90: 305 (5.2%)
Posiciones con F1 < 0.80: 5549 (94.1%)


In [19]:
# Top 10 posiciones mejor reconstruidas
top_indices = np.argsort(f1_array)[-10:][::-1]

print("\nTop 10 posiciones mejor reconstruidas:")
print("="*60)
for idx in top_indices:
    game_idx = idx // 59
    move_idx = idx % 59
    print(f"Partida {game_idx:3d}, Movimiento {move_idx:2d} | F1: {f1_per_position[idx]:.4f}")


Top 10 posiciones mejor reconstruidas:
Partida  58, Movimiento  1 | F1: 1.0000
Partida  58, Movimiento  0 | F1: 1.0000
Partida  56, Movimiento  1 | F1: 1.0000
Partida  56, Movimiento  0 | F1: 1.0000
Partida  59, Movimiento  3 | F1: 1.0000
Partida  59, Movimiento  2 | F1: 1.0000
Partida  59, Movimiento  1 | F1: 1.0000
Partida  59, Movimiento  0 | F1: 1.0000
Partida  55, Movimiento  2 | F1: 1.0000
Partida  55, Movimiento  1 | F1: 1.0000


In [20]:
# Bottom 10 posiciones peor reconstruidas
bottom_indices = np.argsort(f1_array)[:10]

print("\nBottom 10 posiciones peor reconstruidas:")
print("="*60)
for idx in bottom_indices:
    game_idx = idx // 59
    move_idx = idx % 59
    print(f"Partida {game_idx:3d}, Movimiento {move_idx:2d} | F1: {f1_per_position[idx]:.4f}")


Bottom 10 posiciones peor reconstruidas:
Partida  77, Movimiento 34 | F1: 0.0000
Partida  76, Movimiento 40 | F1: 0.0000
Partida  76, Movimiento 45 | F1: 0.0000
Partida  12, Movimiento 45 | F1: 0.0000
Partida  12, Movimiento 48 | F1: 0.0000
Partida  18, Movimiento 57 | F1: 0.0000
Partida  42, Movimiento 58 | F1: 0.0000
Partida  18, Movimiento 58 | F1: 0.0000
Partida  16, Movimiento 14 | F1: 0.0000
Partida  16, Movimiento 18 | F1: 0.0000


## Resumen

**Reconstruction implementado ✓**

- Identificamos features con precisión ≥ 0.95 en train set
- Reconstruimos tableros en test set usando esas features
- Calculamos F1 sobre BSPs de piezas solamente
- Objetivo del paper: ~0.95